In [1]:
import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("GPU is not available to PyTorch.")

Python version: 3.10.21 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:35:39) [MSC v.1942 64 bit (AMD64)]
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
GPU memory: 6.0 GB


### Importing required libraries

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)

from peft import LoraConfig, TaskType, get_peft_model

C:\Users\samru\anaconda3\envs\rag_env\lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


### Reproducability

In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


### Loading the data

In [5]:
# Load the DialogSum dataset specified in the assignment
dataset = load_dataset("neil-code/dialogsum-test")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})


### Inspecting the dataset

In [6]:
# Inspect the dataset
print("Available splits:", list(dataset.keys()))

split_name = list(dataset.keys())[0]
data = dataset[split_name]

print("Selected split:", split_name)
print("Number of examples:", len(data))
print("Columns:", data.column_names)

# Display two examples
for i in range(2):
    print(f"\n{'=' * 80}")
    print(f"Example {i + 1}")
    print("Dialogue:")
    print(data[i]["dialogue"])
    print("\nSummary:")
    print(data[i]["summary"])

Available splits: ['train', 'validation', 'test']
Selected split: train
Number of examples: 1999
Columns: ['id', 'dialogue', 'summary', 'topic']

Example 1
Dialogue:
#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medication

### Converting the dataset to required input format to feed the Model

In [7]:
# Convert each example into input and target fields
def prepare_example(example):
    return {
        "input": "summarize: " + example["dialogue"],
        "target": example["summary"],
    }

processed_data = data.map(prepare_example)

# Display two processed samples
for i in range(2):
    print(f"\n{'=' * 80}")
    print(f"Processed Sample {i + 1}")
    print("Input:")
    print(processed_data[i]["input"])
    print("\nTarget:")
    print(processed_data[i]["target"])


Processed Sample 1
Input:
summarize: #Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?
#Person2#: I found it would be a good idea to get a check-up.
#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.
#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?
#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.
#Person2#: Ok.
#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?
#Person2#: Yes.
#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.
#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.
#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.
#Person2#: Ok, thanks doctor.

Target:
Mr. Smith's getting a

### Baseline Inference by loading the pretrained FLAN-T5-small model

In [8]:
# Load the pretrained model and tokenizer
MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=model_dtype
)

baseline_model = baseline_model.to(DEVICE)
baseline_model.eval()

print("Model loaded:", MODEL_NAME)
print("Model device:", next(baseline_model.parameters()).device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded: google/flan-t5-small
Model device: cuda:0


### Run and Save Baseline Inference Outputs

In [9]:
def generate_summary(model, text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=4,
            do_sample=False
        )

    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)


# Use the same two dialogues for all later comparisons
baseline_examples = []

for i in range(2):
    input_text = processed_data[i]["input"]
    reference_summary = processed_data[i]["target"]
    baseline_summary = generate_summary(baseline_model, input_text)

    baseline_examples.append({
        "dialogue": data[i]["dialogue"],
        "input": input_text,
        "reference_summary": reference_summary,
        "baseline_summary": baseline_summary
    })

    print(f"\n{'=' * 80}")
    print(f"Dialogue {i + 1}")
    print("Baseline summary:")
    print(baseline_summary)


# Save baseline outputs
with open("baseline_outputs.txt", "w", encoding="utf-8") as file:
    for i, example in enumerate(baseline_examples, start=1):
        file.write(f"Dialogue {i}\n")
        file.write(f"Input:\n{example['input']}\n\n")
        file.write(f"Reference summary:\n{example['reference_summary']}\n\n")
        file.write(f"Baseline summary:\n{example['baseline_summary']}\n")
        file.write("\n" + "=" * 80 + "\n")

print("\nBaseline outputs saved to baseline_outputs.txt")


Dialogue 1
Baseline summary:
Get a check-up, Mr. Smith.

Dialogue 2
Baseline summary:
#Person2#: I'm sorry, but I'm not sure if I'll be able to administer the booster.

Baseline outputs saved to baseline_outputs.txt


### Configure and Attach LoRA Adapters

In [10]:
# LoRA hyperparameters
LORA_RANK = 4
LORA_ALPHA = 8
LORA_DROPOUT = 0.1

# Load a fresh copy of the pretrained model for fine-tuning
lora_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=model_dtype
)

lora_base_model = lora_base_model.to(DEVICE)

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q", "v"],
    bias="none"
)

# Attach LoRA adapters
lora_model = get_peft_model(
    lora_base_model,
    lora_config
)

lora_model = lora_model.to(DEVICE)

print("LoRA configuration:")
print(f"Rank (r): {LORA_RANK}")
print(f"Alpha: {LORA_ALPHA}")
print(f"Dropout: {LORA_DROPOUT}")
print("Target modules: q, v")

print("\nParameter counts:")
lora_model.print_trainable_parameters()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LoRA configuration:
Rank (r): 4
Alpha: 8
Dropout: 0.1
Target modules: q, v

Parameter counts:
trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230


### Tokenize DialogSum for Fine-Tuning

In [11]:
# Tokenize the input dialogues and target summaries
def tokenize_function(batch):
    model_inputs = tokenizer(
        batch["input"],
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_data = processed_data.map(
    tokenize_function,
    batched=True,
    remove_columns=processed_data.column_names
)

print(tokenized_data)
print("Tokenized columns:", tokenized_data.column_names)
print("First input token count:", len(tokenized_data[0]["input_ids"]))
print("First target token count:", len(tokenized_data[0]["labels"]))

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1999
})
Tokenized columns: ['input_ids', 'attention_mask', 'labels']
First input token count: 286
First target token count: 48


### Fine-Tune FLAN-T5 with LoRA

In [12]:
# Prepare the model for training
lora_model.config.use_cache = False

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=lora_model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

training_args = TrainingArguments(
    output_dir="lora_r4_output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_data,
    data_collator=data_collator,
    processing_class=tokenizer
)

training_result = trainer.train()

print("\nFine-tuning completed.")
print("Final training loss:", training_result.training_loss)
print("Training time in seconds:", training_result.metrics["train_runtime"])

Step,Training Loss
50,1422179.200000
100,14374993.920000
150,92290385.920000
200,34889743.360000
250,60792345.600000
300,101929359.360000
350,16532766.720000
400,45328486.400000
450,54370892.800000
500,88542771.200000



Fine-tuning completed.
Final training loss: 51047392.448
Training time in seconds: 1877.0386


### Stable LoRA Fine-Tuning

In [14]:
# Stable training configuration
stable_training_args = TrainingArguments(
    output_dir="stable_lora_r4_output",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_steps=50,
    max_grad_norm=1.0,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    fp16=False,
    remove_unused_columns=False
)

# Create the trainer
stable_trainer = Trainer(
    model=stable_lora_model,
    args=stable_training_args,
    train_dataset=tokenized_data,
    data_collator=stable_data_collator,
    processing_class=tokenizer
)

# Start stable fine-tuning
stable_training_result = stable_trainer.train()

print("\nStable fine-tuning completed.")
print("Final training loss:", stable_training_result.training_loss)
print("Training time in seconds:",
      stable_training_result.metrics["train_runtime"])

Step,Training Loss
50,20.601030
100,18.010392
150,15.096062
200,14.285001
250,13.917115
300,13.657555
350,13.763347
400,13.529440
450,13.594231
500,13.075264



Stable fine-tuning completed.
Final training loss: 14.952943725585937
Training time in seconds: 1447.9262


### Saving the Model

In [20]:
import os
import json

SAVE_DIR = "./stable_lora_r4_output"

os.makedirs(SAVE_DIR, exist_ok=True)

# Save trained model / LoRA adapter
stable_trainer.save_model(SAVE_DIR)

# Save tokenizer
tokenizer.save_pretrained(SAVE_DIR)

# Save trainer state
stable_trainer.save_state()

print(f"Model saved to: {SAVE_DIR}")
print("Saved files:")
print(os.listdir(SAVE_DIR))

Model saved to: ./stable_lora_r4_output
Saved files:
['adapter_config.json', 'adapter_model.safetensors', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'trainer_state.json', 'training_args.bin']


### Load the Saved Stable LoRA Adapter

In [12]:
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM

base_model_for_loading = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to(DEVICE)

stable_lora_model = PeftModel.from_pretrained(
    base_model_for_loading,
    "stable_lora_r4_output"
).to(DEVICE)

stable_lora_model.eval()

print("Saved LoRA adapter loaded successfully.")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Saved LoRA adapter loaded successfully.


### Loading the Tokenizer for Inference

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


###  Model Evaluation on Held-Out Test Set

In [15]:
# ============================================================
# MEMORY-SAFE EVALUATION FOR SEQ2SEQ / SUMMARIZATION MODEL
# ============================================================

import torch
import numpy as np
from torch.utils.data import DataLoader
from collections import Counter

# ------------------------------------------------------------
# 1. Get tokenizer from trainer
# ------------------------------------------------------------

# tokenizer = getattr(stable_trainer, "processing_class", None)

# if tokenizer is None:
#     tokenizer = getattr(stable_trainer, "tokenizer", None)

# if tokenizer is None:
#     raise ValueError("Tokenizer could not be found in stable_trainer.")

# print("Tokenizer loaded successfully.")


# ------------------------------------------------------------
# 2. Use the existing held-out test split
# ------------------------------------------------------------

raw_test_dataset = dataset["test"]

print("Test rows:", len(raw_test_dataset))
print("Raw columns:", raw_test_dataset.column_names)


# ------------------------------------------------------------
# 3. Tokenize test data
# ------------------------------------------------------------

def preprocess_test_batch(batch):
    model_inputs = tokenizer(
        batch["dialogue"],
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["summary"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_test_dataset = raw_test_dataset.map(
    preprocess_test_batch,
    batched=True,
    remove_columns=raw_test_dataset.column_names
)

print("Tokenized columns:", tokenized_test_dataset.column_names)


# ------------------------------------------------------------
# 4. Prepare dataset for PyTorch
# ------------------------------------------------------------

tokenized_test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


# ------------------------------------------------------------
# 5. Build small-batch DataLoader
# ------------------------------------------------------------

BATCH_SIZE = 1   # safest for 6 GB GPU

test_loader = DataLoader(
    tokenized_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ------------------------------------------------------------
# 6. Prepare model
# ------------------------------------------------------------

# model = stable_trainer.model
model = stable_lora_model
model.eval()

device = next(model.parameters()).device

print("Model device:", device)

torch.cuda.empty_cache()


# ------------------------------------------------------------
# 7. Generate predictions in memory-safe batches
# ------------------------------------------------------------

generated_summaries = []
reference_summaries = []

with torch.no_grad():

    for batch_idx, batch in enumerate(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            num_beams=1,
            do_sample=False
        )

        # Move generated output to CPU immediately
        generated_ids = generated_ids.cpu()

        # Ground-truth labels
        labels = batch["labels"].cpu()

        # Replace -100 before decoding
        labels = torch.where(
            labels == -100,
            torch.tensor(tokenizer.pad_token_id),
            labels
        )

        # Decode predictions
        batch_generated = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        # Decode references
        batch_references = tokenizer.batch_decode(
            labels,
            skip_special_tokens=True
        )

        generated_summaries.extend(
            [text.strip() for text in batch_generated]
        )

        reference_summaries.extend(
            [text.strip() for text in batch_references]
        )

        # Free GPU memory
        del input_ids
        del attention_mask
        del generated_ids
        del labels

        torch.cuda.empty_cache()

        if (batch_idx + 1) % 25 == 0:
            print(f"Processed {batch_idx + 1} / {len(test_loader)} batches")


print("\nGeneration completed.")
print("Generated summaries:", len(generated_summaries))
print("Reference summaries:", len(reference_summaries))


# ------------------------------------------------------------
# 8. ROUGE helper functions
# ------------------------------------------------------------

def ngrams(tokens, n):
    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]


def rouge_n_f1(prediction, reference, n=1):

    pred_tokens = prediction.lower().split()
    ref_tokens = reference.lower().split()

    pred_ngrams = Counter(ngrams(pred_tokens, n))
    ref_ngrams = Counter(ngrams(ref_tokens, n))

    overlap = sum(
        (pred_ngrams & ref_ngrams).values()
    )

    pred_total = sum(pred_ngrams.values())
    ref_total = sum(ref_ngrams.values())

    if pred_total == 0 or ref_total == 0:
        return 0.0

    precision = overlap / pred_total
    recall = overlap / ref_total

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def lcs_length(a, b):

    previous = [0] * (len(b) + 1)

    for token_a in a:

        current = [0]

        for j, token_b in enumerate(b, start=1):

            if token_a == token_b:
                current.append(
                    previous[j - 1] + 1
                )

            else:
                current.append(
                    max(
                        previous[j],
                        current[j - 1]
                    )
                )

        previous = current

    return previous[-1]


def rouge_l_f1(prediction, reference):

    pred_tokens = prediction.lower().split()
    ref_tokens = reference.lower().split()

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    lcs = lcs_length(
        pred_tokens,
        ref_tokens
    )

    precision = lcs / len(pred_tokens)
    recall = lcs / len(ref_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


# ------------------------------------------------------------
# 9. Calculate ROUGE metrics
# ------------------------------------------------------------

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for pred, ref in zip(
    generated_summaries,
    reference_summaries
):

    rouge1_scores.append(
        rouge_n_f1(
            pred,
            ref,
            n=1
        )
    )

    rouge2_scores.append(
        rouge_n_f1(
            pred,
            ref,
            n=2
        )
    )

    rougeL_scores.append(
        rouge_l_f1(
            pred,
            ref
        )
    )


rouge1 = float(np.mean(rouge1_scores))
rouge2 = float(np.mean(rouge2_scores))
rougeL = float(np.mean(rougeL_scores))


# ------------------------------------------------------------
# 10. Print final results
# ------------------------------------------------------------

print("\n" + "=" * 50)
print("HELD-OUT TEST SET RESULTS")
print("=" * 50)

print(f"Test Examples : {len(generated_summaries)}")
print(f"ROUGE-1 F1   : {rouge1:.4f}")
print(f"ROUGE-2 F1   : {rouge2:.4f}")
print(f"ROUGE-L F1   : {rougeL:.4f}")


# ------------------------------------------------------------
# 11. Show sample predictions
# ------------------------------------------------------------

print("\n" + "=" * 50)
print("SAMPLE PREDICTIONS")
print("=" * 50)

for i in range(min(5, len(generated_summaries))):

    print(f"\nExample {i + 1}")

    print("\nGenerated Summary:")
    print(generated_summaries[i])

    print("\nReference Summary:")
    print(reference_summaries[i])

    print("-" * 50)

Test rows: 499
Raw columns: ['id', 'dialogue', 'summary', 'topic']
Tokenized columns: ['input_ids', 'attention_mask', 'labels']
Model device: cuda:0
Processed 25 / 499 batches
Processed 50 / 499 batches
Processed 75 / 499 batches
Processed 100 / 499 batches
Processed 125 / 499 batches
Processed 150 / 499 batches
Processed 175 / 499 batches
Processed 200 / 499 batches
Processed 225 / 499 batches
Processed 250 / 499 batches
Processed 275 / 499 batches
Processed 300 / 499 batches
Processed 325 / 499 batches
Processed 350 / 499 batches
Processed 375 / 499 batches
Processed 400 / 499 batches
Processed 425 / 499 batches
Processed 450 / 499 batches
Processed 475 / 499 batches

Generation completed.
Generated summaries: 499
Reference summaries: 499

HELD-OUT TEST SET RESULTS
Test Examples : 499
ROUGE-1 F1   : 0.2625
ROUGE-2 F1   : 0.0788
ROUGE-L F1   : 0.2238

SAMPLE PREDICTIONS

Example 1

Generated Summary:
#Person2# is ready to take a dictation for #Person1#. #Person1# will take a dictation

### Base model evaluation

In [16]:
# ============================================================
# BASE MODEL EVALUATION
# Compare base model vs fine-tuned LoRA model
# ============================================================

import torch
import numpy as np
from transformers import AutoModelForSeq2SeqLM
from torch.utils.data import DataLoader
from collections import Counter

# ------------------------------------------------------------
# 1. Replace this with your original base model name
# ------------------------------------------------------------

BASE_MODEL_NAME = MODEL_NAME

# Example:
# BASE_MODEL_NAME = "google/flan-t5-small"


# ------------------------------------------------------------
# 2. Load base model
# ------------------------------------------------------------

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_NAME
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

base_model = base_model.to(device)
base_model.eval()

print("Base model loaded on:", device)


# ------------------------------------------------------------
# 3. Use same tokenized test dataset
# ------------------------------------------------------------

test_loader = DataLoader(
    tokenized_test_dataset,
    batch_size=1,
    shuffle=False
)


# ------------------------------------------------------------
# 4. Generate base-model predictions
# ------------------------------------------------------------

base_generated_summaries = []
base_reference_summaries = []

with torch.no_grad():

    for batch_idx, batch in enumerate(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        generated_ids = base_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            num_beams=1,
            do_sample=False
        )

        generated_ids = generated_ids.cpu()

        labels = batch["labels"].cpu()

        labels = torch.where(
            labels == -100,
            torch.tensor(tokenizer.pad_token_id),
            labels
        )

        generated_text = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        reference_text = tokenizer.batch_decode(
            labels,
            skip_special_tokens=True
        )

        base_generated_summaries.extend(
            [x.strip() for x in generated_text]
        )

        base_reference_summaries.extend(
            [x.strip() for x in reference_text]
        )

        del input_ids
        del attention_mask
        del generated_ids
        del labels

        torch.cuda.empty_cache()

        if (batch_idx + 1) % 50 == 0:
            print(
                f"Processed {batch_idx + 1} / {len(test_loader)}"
            )


print("\nBase model generation complete.")


# ------------------------------------------------------------
# 5. Calculate ROUGE
# ------------------------------------------------------------

base_rouge1_scores = []
base_rouge2_scores = []
base_rougeL_scores = []

for pred, ref in zip(
    base_generated_summaries,
    base_reference_summaries
):

    base_rouge1_scores.append(
        rouge_n_f1(pred, ref, n=1)
    )

    base_rouge2_scores.append(
        rouge_n_f1(pred, ref, n=2)
    )

    base_rougeL_scores.append(
        rouge_l_f1(pred, ref)
    )


base_rouge1 = np.mean(base_rouge1_scores)
base_rouge2 = np.mean(base_rouge2_scores)
base_rougeL = np.mean(base_rougeL_scores)


# ------------------------------------------------------------
# 6. Compare base vs fine-tuned
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("BASE MODEL VS FINE-TUNED MODEL")
print("=" * 55)

print(
    f"{'Metric':<15}"
    f"{'Base':<15}"
    f"{'Fine-Tuned':<15}"
    f"{'Change':<15}"
)

print("-" * 55)

print(
    f"{'ROUGE-1':<15}"
    f"{base_rouge1:<15.4f}"
    f"{rouge1:<15.4f}"
    f"{rouge1 - base_rouge1:<15.4f}"
)

print(
    f"{'ROUGE-2':<15}"
    f"{base_rouge2:<15.4f}"
    f"{rouge2:<15.4f}"
    f"{rouge2 - base_rouge2:<15.4f}"
)

print(
    f"{'ROUGE-L':<15}"
    f"{base_rougeL:<15.4f}"
    f"{rougeL:<15.4f}"
    f"{rougeL - base_rougeL:<15.4f}"
)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Base model loaded on: cuda
Processed 50 / 499
Processed 100 / 499
Processed 150 / 499
Processed 200 / 499
Processed 250 / 499
Processed 300 / 499
Processed 350 / 499
Processed 400 / 499
Processed 450 / 499

Base model generation complete.

BASE MODEL VS FINE-TUNED MODEL
Metric         Base           Fine-Tuned     Change         
-------------------------------------------------------
ROUGE-1        0.0784         0.2625         0.1842         
ROUGE-2        0.0146         0.0788         0.0642         
ROUGE-L        0.0689         0.2238         0.1549         


### Base Model vs Fine-Tuned Model Comparison

The fine-tuned FLAN-T5 model outperformed the original base model on all ROUGE metrics. ROUGE-1 increased from 0.0784 to 0.2625, ROUGE-2 increased from 0.0146 to 0.0788, and ROUGE-L increased from 0.0689 to 0.2238. This demonstrates that LoRA fine-tuning improved the model's ability to generate summaries similar to the reference summaries.

### check output diversity

In [17]:
from collections import Counter

prediction_counts = Counter(generated_summaries)

total_predictions = len(generated_summaries)
unique_predictions = len(prediction_counts)

unique_ratio = unique_predictions / total_predictions

print("=" * 55)
print("OUTPUT DIVERSITY ANALYSIS")
print("=" * 55)

print(f"Total predictions  : {total_predictions}")
print(f"Unique predictions : {unique_predictions}")
print(f"Unique ratio       : {unique_ratio:.2%}")

print("\nMost Frequently Generated Summaries")
print("=" * 55)

for rank, (text, count) in enumerate(
    prediction_counts.most_common(10),
    start=1
):
    percentage = (count / total_predictions) * 100

    print(f"\n#{rank}")
    print(f"Generated {count} times ({percentage:.2f}%)")
    print(text)
    print("-" * 55)

OUTPUT DIVERSITY ANALYSIS
Total predictions  : 499
Unique predictions : 166
Unique ratio       : 33.27%

Most Frequently Generated Summaries

#1
Generated 6 times (1.20%)
#Person1# and #Person2# are trying to get more anger from Mike.
-------------------------------------------------------

#2
Generated 3 times (0.60%)
#Person2# is ready to take a dictation for #Person1#. #Person1# will take a dictation for #Person2#.
-------------------------------------------------------

#3
Generated 3 times (0.60%)
#Person1# is stuck in traffic again. #Person2# thinks it's better to take the subway to work. #Person2# thinks biking to work would be better for the environment.
-------------------------------------------------------

#4
Generated 3 times (0.60%)
#Person1# and #Person2# are having a separation for 2 months and filed for divorce.
-------------------------------------------------------

#5
Generated 3 times (0.60%)
Brian and Brian are happy to have a dance with Brian and Brian. Brian's n

### Output Diversity Analysis

The fine-tuned model generated 166 unique summaries out of 499 test examples, giving a unique-output ratio of 33.27%. No single summary dominated the predictions; the most frequent summary appeared only six times. However, some outputs contained repeated phrases, incorrect details, or unnatural wording. This indicates that fine-tuning improved similarity to the reference summaries but did not completely eliminate generation-quality issues.

### error analysis

In [18]:
# ============================================================
# ERROR ANALYSIS - FINE-TUNED MODEL
# ============================================================

import pandas as pd
import numpy as np
from collections import Counter


# ------------------------------------------------------------
# 1. Create evaluation dataframe
# ------------------------------------------------------------

results_df = pd.DataFrame({
    "dialogue": dataset["test"]["dialogue"],
    "reference": reference_summaries,
    "prediction": generated_summaries
})


# ------------------------------------------------------------
# 2. Count how often each prediction occurs
# ------------------------------------------------------------

prediction_frequency = Counter(generated_summaries)

results_df["prediction_frequency"] = (
    results_df["prediction"]
    .map(prediction_frequency)
)


# ------------------------------------------------------------
# 3. Detect repeated sentences inside a summary
# ------------------------------------------------------------

def has_repeated_sentence(text):

    sentences = [
        sentence.strip()
        for sentence in text.split(".")
        if sentence.strip()
    ]

    return len(sentences) != len(set(sentences))


results_df["repeated_sentence"] = (
    results_df["prediction"]
    .apply(has_repeated_sentence)
)


# ------------------------------------------------------------
# 4. Detect suspiciously short outputs
# ------------------------------------------------------------

results_df["prediction_words"] = (
    results_df["prediction"]
    .str.split()
    .str.len()
)

results_df["reference_words"] = (
    results_df["reference"]
    .str.split()
    .str.len()
)

results_df["too_short"] = (
    results_df["prediction_words"]
    <
    (0.4 * results_df["reference_words"])
)


# ------------------------------------------------------------
# 5. Detect suspiciously long outputs
# ------------------------------------------------------------

results_df["too_long"] = (
    results_df["prediction_words"]
    >
    (2.0 * results_df["reference_words"])
)


# ------------------------------------------------------------
# 6. Flag predictions reused across multiple inputs
# ------------------------------------------------------------

results_df["duplicate_prediction"] = (
    results_df["prediction_frequency"] > 1
)


# ------------------------------------------------------------
# 7. Add individual ROUGE-L score
# ------------------------------------------------------------

results_df["rougeL"] = [
    rouge_l_f1(pred, ref)
    for pred, ref in zip(
        results_df["prediction"],
        results_df["reference"]
    )
]


# ------------------------------------------------------------
# 8. Print overall error statistics
# ------------------------------------------------------------

print("=" * 60)
print("FINE-TUNED MODEL ERROR ANALYSIS")
print("=" * 60)

print(f"Test examples              : {len(results_df)}")

print(
    f"Unique predictions         : "
    f"{results_df['prediction'].nunique()}"
)

print(
    f"Repeated-output examples   : "
    f"{results_df['duplicate_prediction'].sum()}"
)

print(
    f"Repeated-sentence examples : "
    f"{results_df['repeated_sentence'].sum()}"
)

print(
    f"Suspiciously short         : "
    f"{results_df['too_short'].sum()}"
)

print(
    f"Suspiciously long          : "
    f"{results_df['too_long'].sum()}"
)

print(
    f"Mean prediction length     : "
    f"{results_df['prediction_words'].mean():.1f} words"
)

print(
    f"Mean reference length      : "
    f"{results_df['reference_words'].mean():.1f} words"
)

print(
    f"Mean ROUGE-L               : "
    f"{results_df['rougeL'].mean():.4f}"
)


# ------------------------------------------------------------
# 9. Show worst predictions
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("10 LOWEST ROUGE-L EXAMPLES")
print("=" * 60)

worst_examples = (
    results_df
    .sort_values("rougeL")
    .head(10)
)

for idx, row in worst_examples.iterrows():

    print("\n" + "=" * 60)

    print("DIALOGUE:")
    print(row["dialogue"][:1000])

    print("\nMODEL:")
    print(row["prediction"])

    print("\nREFERENCE:")
    print(row["reference"])

    print(
        f"\nROUGE-L: "
        f"{row['rougeL']:.4f}"
    )


# ------------------------------------------------------------
# 10. Show repeated-output examples
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("MOST REPEATED OUTPUTS")
print("=" * 60)

repeated = (
    results_df[
        results_df["duplicate_prediction"]
    ]
    .sort_values(
        "prediction_frequency",
        ascending=False
    )
    .head(10)
)

for idx, row in repeated.iterrows():

    print("\n" + "=" * 60)

    print(
        f"OUTPUT USED "
        f"{row['prediction_frequency']} TIMES"
    )

    print("\nDIALOGUE:")
    print(row["dialogue"][:700])

    print("\nMODEL:")
    print(row["prediction"])

    print("\nREFERENCE:")
    print(row["reference"])


# ------------------------------------------------------------
# 11. Save results for later inspection
# ------------------------------------------------------------

results_df.to_csv(
    "fine_tuned_error_analysis.csv",
    index=False
)

print(
    "\nSaved detailed analysis to: "
    "fine_tuned_error_analysis.csv"
)

FINE-TUNED MODEL ERROR ANALYSIS
Test examples              : 499
Unique predictions         : 166
Repeated-output examples   : 498
Repeated-sentence examples : 45
Suspiciously short         : 36
Suspiciously long          : 45
Mean prediction length     : 21.0 words
Mean reference length      : 19.3 words
Mean ROUGE-L               : 0.2238


10 LOWEST ROUGE-L EXAMPLES

DIALOGUE:
#Person1#: Yeah. Just pull on this strip. Then peel off the back.
#Person2#: You might make a few enemies this way.
#Person1#: If they don't think this is fun, they're not meant to be our friends.
#Person2#: You mean your friends. I think it's cruel.
#Person1#: Yeah. But it's fun. Look at those two ugly old ladies. . . or are they men?
#Person2#: Hurry! Get a shot!. . . Hand it over!
#Person1#: I knew you'd come around. . .

MODEL:
#Person1# pulls on a strip and peels off the back.

REFERENCE:
#Person2# at first thinks #Person1#'s behaviour cruel but finally joins #Person1#.

ROUGE-L: 0.0000

DIALOGUE:
#Person

### Check Whether Duplicate Predictions Come From Different Dialogues

In [19]:
# ============================================================
# DUPLICATE PREDICTION INVESTIGATION
# ============================================================

import pandas as pd
from collections import Counter

# Build results table if it doesn't already exist
results_df = pd.DataFrame({
    "dialogue": dataset["test"]["dialogue"],
    "reference": reference_summaries,
    "prediction": generated_summaries
})

# Count prediction frequency
prediction_counts = Counter(results_df["prediction"])

# Predictions appearing more than once
duplicate_predictions = {
    pred: count
    for pred, count in prediction_counts.items()
    if count > 1
}

print("=" * 70)
print("DUPLICATE PREDICTION INVESTIGATION")
print("=" * 70)

print(f"Total test examples       : {len(results_df)}")
print(f"Unique predictions        : {results_df['prediction'].nunique()}")
print(f"Repeated prediction types : {len(duplicate_predictions)}")


# ------------------------------------------------------------
# Examine the 5 most frequently repeated predictions
# ------------------------------------------------------------

most_common = sorted(
    duplicate_predictions.items(),
    key=lambda x: x[1],
    reverse=True
)[:5]

for rank, (prediction, count) in enumerate(most_common, 1):

    matching_rows = results_df[
        results_df["prediction"] == prediction
    ]

    print("\n" + "=" * 70)
    print(f"REPEATED OUTPUT #{rank}")
    print(f"Generated {count} times")
    print("=" * 70)

    print("\nMODEL OUTPUT:")
    print(prediction)

    for example_number, (_, row) in enumerate(
        matching_rows.iterrows(),
        1
    ):

        print("\n" + "-" * 70)
        print(f"INPUT {example_number}")
        print("-" * 70)

        print("\nDialogue:")
        print(row["dialogue"])

        print("\nCorrect Reference Summary:")
        print(row["reference"])

DUPLICATE PREDICTION INVESTIGATION
Total test examples       : 499
Unique predictions        : 166
Repeated prediction types : 165

REPEATED OUTPUT #1
Generated 6 times

MODEL OUTPUT:
#Person1# and #Person2# are trying to get more anger from Mike.

----------------------------------------------------------------------
INPUT 1
----------------------------------------------------------------------

Dialogue:
#Person1#: OK, that's a cut! Let's start from the beginning, everyone.
#Person2#: What was the problem that time?
#Person1#: The feeling was all wrong, Mike. She is telling you that she doesn't want to see you any more, but I want to get more anger from you. You're acting hurt and sad, but that's not how your character would act in this situation.
#Person2#: But Jason and Laura have been together for three years. Don't you think his reaction would be one of both anger and sadness?
#Person1#: At this point, no. I think he would react the way most guys would, and then later on, we woul

### check for train/test leakage

In [20]:
# ============================================================
# TRAIN / TEST LEAKAGE CHECK
# ============================================================

train_dialogues = set(dataset["train"]["dialogue"])
test_dialogues = set(dataset["test"]["dialogue"])

train_summaries = set(dataset["train"]["summary"])
test_summaries = set(dataset["test"]["summary"])


dialogue_overlap = train_dialogues.intersection(test_dialogues)
summary_overlap = train_summaries.intersection(test_summaries)


print("=" * 60)
print("TRAIN / TEST LEAKAGE CHECK")
print("=" * 60)

print(
    f"Exact dialogue overlap : "
    f"{len(dialogue_overlap)}"
)

print(
    f"Exact summary overlap  : "
    f"{len(summary_overlap)}"
)


if len(dialogue_overlap) == 0:
    print("\n✓ No exact dialogue leakage detected.")
else:
    print("\n⚠ Exact training dialogues exist in test data.")


if len(summary_overlap) == 0:
    print("✓ No exact summary leakage detected.")
else:
    print("⚠ Exact training summaries exist in test data.")

TRAIN / TEST LEAKAGE CHECK
Exact dialogue overlap : 0
Exact summary overlap  : 0

✓ No exact dialogue leakage detected.
✓ No exact summary leakage detected.


### Train the LoRA Model with Rank r = 16

In [22]:
r16_trainer = Trainer(
    model=r16_model,
    args=r16_training_args,
    train_dataset=tokenized_r16_dataset,
    processing_class=tokenizer,
    data_collator=r16_data_collator
)

print("r=16 Trainer created successfully.")

r=16 Trainer created successfully.


In [23]:
print("Starting LoRA r=16 training...")

r16_train_result = r16_trainer.train()

print("r=16 training completed.")

r16_trainer.save_model("lora_r16_output")
tokenizer.save_pretrained("lora_r16_output")

print("Saved r=16 adapter to: lora_r16_output")

Starting LoRA r=16 training...


Step,Training Loss
50,10.272559
100,7.595305
150,7.174207
200,6.901099
250,6.621743
300,6.421418
350,6.483375
400,6.403085
450,6.306946
500,6.543156


r=16 training completed.
Saved r=16 adapter to: lora_r16_output


### r = 16 Evaluation

In [24]:
# Save the completed r=4 results
r4_rouge1 = rouge1
r4_rouge2 = rouge2
r4_rougeL = rougeL

# Prepare the trained r=16 model
r16_model.eval()

print("r=4 results saved.")
print("r=16 model ready for evaluation.")

r=4 results saved.
r=16 model ready for evaluation.


In [26]:
# ============================================================
# EVALUATE LoRA r = 16 MODEL
# ============================================================

r16_model.eval()

r16_generated_summaries = []
r16_reference_summaries = []

device = next(r16_model.parameters()).device

print("Evaluating r=16 model on:", device)

with torch.no_grad():

    for batch_idx, batch in enumerate(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        generated_ids = r16_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            num_beams=1,
            do_sample=False
        )

        generated_ids = generated_ids.cpu()

        labels = batch["labels"].cpu()

        labels = torch.where(
        labels == -100,
        torch.full_like(
            labels,
            tokenizer.pad_token_id
        ),
        labels
        )

        generated_text = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        reference_text = tokenizer.batch_decode(
            labels,
            skip_special_tokens=True
        )

        r16_generated_summaries.extend(
            [text.strip() for text in generated_text]
        )

        r16_reference_summaries.extend(
            [text.strip() for text in reference_text]
        )

        del input_ids
        del attention_mask
        del generated_ids
        del labels

        torch.cuda.empty_cache()

        if (batch_idx + 1) % 50 == 0:
            print(
                f"Processed {batch_idx + 1} / "
                f"{len(test_loader)}"
            )

print("\nGeneration completed.")

# ------------------------------------------------------------
# Calculate r=16 ROUGE scores
# ------------------------------------------------------------

r16_rouge1_scores = []
r16_rouge2_scores = []
r16_rougeL_scores = []

for prediction, reference in zip(
    r16_generated_summaries,
    r16_reference_summaries
):

    r16_rouge1_scores.append(
        rouge_n_f1(
            prediction,
            reference,
            n=1
        )
    )

    r16_rouge2_scores.append(
        rouge_n_f1(
            prediction,
            reference,
            n=2
        )
    )

    r16_rougeL_scores.append(
        rouge_l_f1(
            prediction,
            reference
        )
    )

r16_rouge1 = float(np.mean(r16_rouge1_scores))
r16_rouge2 = float(np.mean(r16_rouge2_scores))
r16_rougeL = float(np.mean(r16_rougeL_scores))

# ------------------------------------------------------------
# Print r=16 results
# ------------------------------------------------------------

print("\n" + "=" * 50)
print("LoRA r=16 TEST RESULTS")
print("=" * 50)

print(f"Test Examples : {len(r16_generated_summaries)}")
print(f"ROUGE-1 F1    : {r16_rouge1:.4f}")
print(f"ROUGE-2 F1    : {r16_rouge2:.4f}")
print(f"ROUGE-L F1    : {r16_rougeL:.4f}")

Evaluating r=16 model on: cuda:0
Processed 50 / 499
Processed 100 / 499
Processed 150 / 499
Processed 200 / 499
Processed 250 / 499
Processed 300 / 499
Processed 350 / 499
Processed 400 / 499
Processed 450 / 499

Generation completed.

LoRA r=16 TEST RESULTS
Test Examples : 499
ROUGE-1 F1    : 0.3131
ROUGE-2 F1    : 0.1027
ROUGE-L F1    : 0.2599


### Compare r = 4 and r = 16

In [27]:
print("=" * 60)
print("LoRA RANK COMPARISON")
print("=" * 60)

print(
    f"{'Metric':<15}"
    f"{'r=4':<15}"
    f"{'r=16':<15}"
    f"{'Improvement':<15}"
)

print("-" * 60)

print(
    f"{'ROUGE-1':<15}"
    f"{r4_rouge1:<15.4f}"
    f"{r16_rouge1:<15.4f}"
    f"{r16_rouge1 - r4_rouge1:<15.4f}"
)

print(
    f"{'ROUGE-2':<15}"
    f"{r4_rouge2:<15.4f}"
    f"{r16_rouge2:<15.4f}"
    f"{r16_rouge2 - r4_rouge2:<15.4f}"
)

print(
    f"{'ROUGE-L':<15}"
    f"{r4_rougeL:<15.4f}"
    f"{r16_rougeL:<15.4f}"
    f"{r16_rougeL - r4_rougeL:<15.4f}"
)

LoRA RANK COMPARISON
Metric         r=4            r=16           Improvement    
------------------------------------------------------------
ROUGE-1        0.2625         0.3131         0.0505         
ROUGE-2        0.0788         0.1027         0.0239         
ROUGE-L        0.2238         0.2599         0.0361         


### LoRA Rank Experiment Results

### LoRA Rank Experiment: r=4 vs r=16

The r=16 LoRA model performed better than the r=4 model on all evaluation metrics.

| Metric | r=4 | r=16 | Improvement |
|---|---:|---:|---:|
| ROUGE-1 | 0.2625 | 0.3131 | +0.0506 |
| ROUGE-2 | 0.0788 | 0.1027 | +0.0239 |
| ROUGE-L | 0.2238 | 0.2599 | +0.0361 |

The higher LoRA rank gave the model more trainable parameters and allowed it to learn the DialogSum summarization patterns more effectively. Therefore, r=16 produced better summary quality than r=4 in this experiment.

### Compare Base Model and r = 16 Model on Two Dialogues

In [28]:
# ============================================================
# BASE MODEL VS r=16 MODEL: TWO-DIALOGUE COMPARISON
# ============================================================

import torch
from transformers import AutoModelForSeq2SeqLM

# Use the same two dialogues for both models
comparison_dialogues = dataset["test"]["dialogue"][:2]

device = next(r16_model.parameters()).device

# Load original base model
comparison_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
).to(device)

comparison_base_model.eval()
r16_model.eval()


def generate_summary(model, dialogue):

    inputs = tokenizer(
        dialogue,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=1,
            do_sample=False
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).strip()


print("=" * 70)
print("BASE MODEL VS r=16 FINE-TUNED MODEL")
print("=" * 70)

for i, dialogue in enumerate(comparison_dialogues):

    base_summary = generate_summary(
        comparison_base_model,
        dialogue
    )

    finetuned_summary = generate_summary(
        r16_model,
        dialogue
    )

    print(f"\nDIALOGUE {i + 1}")
    print("-" * 70)
    print(dialogue)

    print("\nBASE MODEL SUMMARY:")
    print(base_summary)

    print("\nr=16 FINE-TUNED SUMMARY:")
    print(finetuned_summary)

    print("=" * 70)

# Free the temporary base model
del comparison_base_model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


BASE MODEL VS r=16 FINE-TUNED MODEL

DIALOGUE 1
----------------------------------------------------------------------
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't

### Before-and-After Inference Comparison

The original base model generated a generic response that did not summarize the dialogue. The r=16 fine-tuned model correctly recognized that the conversation involved taking a dictation, but it incorrectly assigned the speaker roles and repeated part of the summary. This demonstrates that LoRA fine-tuning improved the model's overall summarization performance, although some individual predictions still contain factual and repetition errors.